In [1]:
# %% [markdown]
# # 01 — Ridge Baseline
#
# Linear baseline for all experiments. Establishes the floor that
# nonlinear models (KAN, MLP) must beat to justify their complexity.
#
# Runs:
# - 4 splits × 2 feature sets × 2 target types = 16 models
# - Grid search over regularisation strength on validation set
# - Evaluates on train, val, and test
# - Saves all predictions and metrics
#
# Target notes:
#   binary:     y_binary (minret_5d_pct < -2), LogisticRegression,
#               select by validation AUC. AUC is used uniformly as the
#               binary selection metric across every model in this
#               project (Ridge, Dense/Sparse MLP, Dense/Sparse KAN) --
#               a per-model split (e.g. Brier for Ridge only) was
#               considered and rejected in favour of one consistent,
#               statable methodology across the whole comparison.
#   continuous: minret_5d_pct (raw five-day block minimum, percentage points),
#               Ridge regression, select by R²
#               Derived AUC uses -y_pred as ranking score against y_binary.
#               No thresholding of predictions — the -2% threshold lives in the
#               binary label, and ranking is what the derived AUC measures.
#
# Reads the Stage 5 splits directly rather than through data_utils, since the
# continuous target changed from an expanding z-score to raw percentage points.
#
# Expected runtime: < 2 minutes total (+ ~1 min per split for backtests).
# %%
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
import warnings
import numpy as np
import pandas as pd
import time
from pathlib import Path
from sklearn.linear_model import LogisticRegression, Ridge
from evaluation import (
    compute_binary_metrics,
    compute_continuous_metrics,
    derive_binary_from_continuous,
    save_predictions,
    load_predictions,
    compute_calibration,
    run_full_backtest,
)

# ═══════════════════════════════════════════════════════════════════════════════
# CONFIGURATION
# ═══════════════════════════════════════════════════════════════════════════════

# ── Paths — UPDATE THESE for your environment ──
SPLITS_DIR  = (Path("../../..") / "Data" / "Data_Collection" / "Final"
               / "Stage_5_Model_Ready" / "04_splits")
RESULTS_DIR = Path("../../..") / "Data" / "Results" / "Dense_vs_Sparse_KAN" / "Ridge"
# Colab:
# SPLITS_DIR  = Path("/content/drive/MyDrive/Thesis/Data/Data_Collection/Final/Stage_5_Model_Ready/04_splits")
# RESULTS_DIR = Path("/content/drive/MyDrive/Thesis/Data/Results/Dense_vs_Sparse_KAN")

SPLITS       = ["Split_A", "Split_B", "Split_C", "Split_D"]
FEATURE_SETS = ["agg_full_moments", "agg_means"]
TARGET_TYPES = ["binary", "continuous"]

# ── Meta columns per dataset ──
# Everything not listed here is a model feature, including the five binary
# regime indicators, which pass through un-z-scored.
META = {
    "agg_means":        ["date", "target_daily_return", "minret_5d_pct", "y_binary"],
    "agg_full_moments": ["date", "target_daily_return", "minret_5d_pct", "y_binary"],
    "panel":            ["permno", "date", "dlyret", "dlycap",
                         "minret_5d_pct", "y_binary"],
}

# ── Ridge hyperparameter grids ──
# Logistic: C = inverse regularisation (higher = less regularisation)
# Linear:   alpha = regularisation strength (higher = more regularisation)
#
# Neither grid needs rescaling for the change of target. Scaling y by a constant
# c scales the ridge solution by exactly c, so predictions scale by c and R² is
# unchanged, leaving the optimal alpha identical. The coefficients themselves are
# now in percentage points of drawdown rather than standardised units.
C_GRID     = [0.00001, 0.00005, 0.0001, 0.0005, 0.001, 0.01, 0.1, 1.0, 10.0, 100.0]
ALPHA_GRID = [1.0, 10.0, 100.0, 1000.0, 10000.0, 30000.0, 50000.0, 75000.0,
              100000.0, 200000.0, 500000.0, 1000000.0]
CLIP_RANGE = 5.0   # must match 05_splits.ipynb


# %% [markdown]
# ## Helper Functions
# %%
def load_split(split_name: str, dataset: str, splits_dir: Path = SPLITS_DIR) -> dict:
    """
    Load one split of one dataset from the Stage 5 output.

    The continuous target is minret_5d_pct, the raw five-day block minimum in
    percentage points, not the expanding z-score the previous pipeline used.
    Nothing is standardised here: the feature matrix arrives already clipped to
    ±CLIP_RANGE and cast to float32 by 05_splits.ipynb.

    target_daily_return is pre-shifted, so row t holds the return earned on day
    t+1. A signal formed at t and applied to this column is therefore correctly
    aligned for the backtest with no further shifting.
    """
    out, meta = {}, META[dataset]
    for part in ("train", "val", "test"):
        df = pd.read_parquet(splits_dir / split_name / f"{dataset}_{part}.parquet")
        feats = [c for c in df.columns if c not in meta]
        out[f"X_{part}"]      = df[feats].to_numpy(dtype=np.float32)
        out[f"y_{part}"]      = df["y_binary"].to_numpy(dtype=np.float32)
        out[f"minret_{part}"] = df["minret_5d_pct"].to_numpy(dtype=np.float32)
        out[f"dates_{part}"]  = df["date"].reset_index(drop=True)
        rcol = "dlyret" if dataset == "panel" else "target_daily_return"
        out[f"returns_{part}"] = df[rcol].to_numpy(dtype=np.float64)
        if dataset == "panel":
            out[f"permno_{part}"] = df["permno"].to_numpy()
    out["feature_cols"] = feats
    return out


def clip_features(X: np.ndarray, clip_range: float = CLIP_RANGE) -> np.ndarray:
    """
    Clip features to [-clip_range, clip_range].
    Features arrive already clipped from 05_splits.ipynb, so this is a safety
    net only — it should be a no-op in practice.
    """
    return np.clip(X, -clip_range, clip_range)


def grid_search_logistic_ridge(X_train, y_train, X_val, y_val):
    """
    Grid search over C for LogisticRegression with L2 penalty.

    Selects the C that MAXIMISES validation AUC. AUC is used uniformly as
    the binary selection metric across every model in this project
    (Ridge, Dense/Sparse MLP, Dense/Sparse KAN — see training.py
    docstring for the neural-network side of this decision). A per-model
    split (e.g. Brier score for Ridge specifically, on the grounds that
    it may be lower-variance under small crash-event counts) was
    considered and rejected: that argument was never actually established
    once autocorrelation/volatility clustering — which depresses
    effective sample size for BOTH metrics similarly, not just AUC — is
    accounted for. Using AUC everywhere supports one clean, statable
    methodological claim: "binary models were selected on validation AUC
    throughout."

    Test AUC remains the headline metric and is reported below. Val Brier
    is also computed and reported for reference, not used for selection.

    Note on class_weight: 'balanced' reweights the loss, which would have
    distorted the probabilities a Brier-based selection depends on. Since
    selection is now rank-based (AUC), this reweighting poses no such
    risk — AUC is invariant to any monotonic rescaling of the predicted
    probabilities.
    """
    best_auc   = -np.inf
    best_C     = 1.0
    best_model = None

    for C in C_GRID:
        model = LogisticRegression(
            C=C,
            class_weight="balanced",
            solver="lbfgs",
            max_iter=10000,
            random_state=42,
        )
        model.fit(X_train, y_train)
        y_prob  = model.predict_proba(X_val)[:, 1]
        metrics = compute_binary_metrics(y_val, y_prob)
        if metrics["auc"] > best_auc:
            best_auc   = metrics["auc"]
            best_C     = C
            best_model = model

    # report the Brier score of the selected model for reference, not for
    # selection
    val_brier = compute_binary_metrics(
        y_val, best_model.predict_proba(X_val)[:, 1]
    )["brier"]

    return best_model, {"C": best_C,
                        "best_val_auc": best_auc,
                        "best_val_brier": val_brier}


def grid_search_linear_ridge(X_train, y_train, X_val, y_val):
    """
    Grid search over alpha for Ridge regression.

    y_train / y_val are minret_5d_pct, the raw five-day block minimum in
    percentage points, NOT z-scores. Selection criterion is validation R²
    (maximise), which is equivalent to minimising MSE on a fixed dataset but
    consistent with how all other continuous models are selected.

    Ill-conditioned warnings are suppressed — Ridge regularisation handles the
    p >> n regime by design.
    """
    best_r2    = -np.inf
    best_alpha = 1.0
    best_model = None

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        for alpha in ALPHA_GRID:
            model = Ridge(alpha=alpha, random_state=42)
            model.fit(X_train, y_train)
            y_pred  = model.predict(X_val)
            metrics = compute_continuous_metrics(y_val, y_pred)
            if metrics["r2"] > best_r2:
                best_r2    = metrics["r2"]
                best_alpha = alpha
                best_model = model

    return best_model, {"alpha": best_alpha, "best_val_r2": best_r2}


def evaluate_and_save_binary(
    model, data, split_name, feature_set, results_dir, hyperparameters=None,
):
    """Evaluate logistic Ridge on train/val/test and save everything."""
    model_name = f"ridge_{feature_set}"
    results    = {}
    for part in ["train", "val", "test"]:
        X       = clip_features(data[f"X_{part}"])
        y       = data[f"y_{part}"]
        dates   = data[f"dates_{part}"]
        returns = data[f"returns_{part}"]

        y_prob  = model.predict_proba(X)[:, 1]
        metrics = compute_binary_metrics(y, y_prob)
        metrics["y_true"] = y
        metrics["y_prob"] = y_prob

        cal = compute_calibration(y, y_prob)
        metrics["ece"] = cal["ece"]

        save_predictions(
            model_name=model_name,
            split_name=split_name,
            target_type="binary",
            part=part,
            dates=dates,
            returns=returns,
            metrics=metrics,
            hyperparameters=hyperparameters if part == "test" else None,
            results_dir=results_dir,
        )
        results[part] = metrics
    return results


def evaluate_and_save_continuous(
    model, data, split_name, feature_set, results_dir, hyperparameters=None,
):
    """
    Evaluate linear Ridge on train/val/test and save everything.

    y_true and y_pred are both minret_5d_pct in percentage points, so MSE is in
    percent² and is directly comparable to the volatility baseline.

    Derived AUC uses -y_pred as the ranking score against y_binary labels.
    y_binary is passed explicitly to derive_binary_from_continuous and to
    save_predictions so the saved parquet contains the y_true_binary column.
    """
    model_name = f"ridge_{feature_set}"
    results    = {}
    for part in ["train", "val", "test"]:
        X        = clip_features(data[f"X_{part}"])
        y_true   = data[f"minret_{part}"]      # percentage points
        y_binary = data[f"y_{part}"]           # binary labels (0/1)
        dates    = data[f"dates_{part}"]
        returns  = data[f"returns_{part}"]

        y_pred  = model.predict(X).ravel()
        metrics = compute_continuous_metrics(y_true, y_pred)

        # Derived AUC: pass y_binary explicitly so derive_binary_from_continuous
        # uses the correct 0/1 labels rather than trying to threshold the target.
        derived = derive_binary_from_continuous(y_binary, y_pred)
        metrics.update(derived)
        metrics["y_true"] = y_true
        metrics["y_pred"] = y_pred

        save_predictions(
            model_name=model_name,
            split_name=split_name,
            target_type="continuous",
            part=part,
            dates=dates,
            returns=returns,
            metrics=metrics,
            hyperparameters=hyperparameters if part == "test" else None,
            results_dir=results_dir,
            # Pass binary labels so parquet contains y_true_binary column
            y_true_binary=y_binary,
        )
        results[part] = metrics
    return results


# %% [markdown]
# ## Run All Experiments
# %%
print("=" * 70)
print("  RIDGE BASELINE: 4 Splits × 2 Feature Sets × 2 Targets")
print("  Continuous target: minret_5d_pct (percentage points, NOT z-scored)")
print("  Continuous selection: maximise val R²")
print("  Binary selection: maximise val AUC (uniform selection metric across")
print("  every model in this project — Ridge, Dense/Sparse MLP, Dense/Sparse KAN)")
print("  Derived AUC: roc_auc_score(y_binary, -y_pred) — rank-based, no threshold")
print("=" * 70)

all_results = []
start_time  = time.time()

for feature_set in FEATURE_SETS:
    for split_name in SPLITS:
        data = load_split(split_name, feature_set, SPLITS_DIR)
        X_train = clip_features(data["X_train"])
        X_val   = clip_features(data["X_val"])

        # ── BINARY TARGET ──
        print(f"\n  {feature_set} / {split_name} / binary")
        model_bin, info_bin = grid_search_logistic_ridge(
            X_train, data["y_train"], X_val, data["y_val"]
        )
        results_bin = evaluate_and_save_binary(
            model_bin, data, split_name, feature_set, RESULTS_DIR,
            hyperparameters=info_bin,
        )
        print(f"    Best C={info_bin['C']:.5f}  |  "
              f"Val AUC={info_bin['best_val_auc']:.4f}  |  "
              f"Val Brier={info_bin['best_val_brier']:.4f}  |  "
              f"Test AUC={results_bin['test']['auc']:.4f}  |  "
              f"Gap={results_bin['train']['auc'] - results_bin['test']['auc']:+.4f}")

        all_results.append({
            "feature_set": feature_set,
            "split":       split_name,
            "target":      "binary",
            "val_metric":  info_bin["best_val_auc"],
            "val_brier":   info_bin["best_val_brier"],
            "test_auc":    results_bin["test"]["auc"],
            "test_brier":  results_bin["test"]["brier"],
            "train_auc":   results_bin["train"]["auc"],
            "best_param":  f"C={info_bin['C']}",
        })

        # ── CONTINUOUS TARGET ──
        print(f"  {feature_set} / {split_name} / continuous")
        model_cont, info_cont = grid_search_linear_ridge(
            X_train, data["minret_train"], X_val, data["minret_val"]
        )
        results_cont = evaluate_and_save_continuous(
            model_cont, data, split_name, feature_set, RESULTS_DIR,
            hyperparameters=info_cont,
        )
        print(f"    Best α={info_cont['alpha']:.0f}  |  "
              f"Val R²={info_cont['best_val_r2']:.4f}  |  "
              f"Test R²={results_cont['test']['r2']:.4f}  |  "
              f"Test MSE={results_cont['test']['mse']:.4f}  |  "
              f"Derived AUC={results_cont['test']['derived_auc']:.4f}  |  "
              f"Pred std={results_cont['test']['pred_std']:.4f}")

        all_results.append({
            "feature_set":      feature_set,
            "split":            split_name,
            "target":           "continuous",
            "val_metric":       info_cont["best_val_r2"],
            "test_r2":          results_cont["test"]["r2"],
            "test_mse":         results_cont["test"]["mse"],
            "test_derived_auc": results_cont["test"]["derived_auc"],
            "test_pred_std":    results_cont["test"]["pred_std"],
            "train_r2":         results_cont["train"]["r2"],
            "best_param":       f"α={info_cont['alpha']}",
        })

total_time = time.time() - start_time
print(f"\n  Total time: {total_time:.1f}s")

# %% [markdown]
# ## Results Summary
# %%
results_df = pd.DataFrame(all_results)
binary_df  = results_df[results_df["target"] == "binary"]
cont_df    = results_df[results_df["target"] == "continuous"]

# ── Binary AUC ──
print("\n" + "=" * 70)
print("  BINARY TARGET — Test AUC (selection metric)")
print("=" * 70 + "\n")
pivot_bin = binary_df.pivot_table(
    index="feature_set", columns="split", values="test_auc"
)
pivot_bin["Mean"] = pivot_bin.mean(axis=1)
print(pivot_bin.round(4).to_string())

# ── Binary Brier (reference only, not the selection metric) ──
print("\n" + "=" * 70)
print("  BINARY TARGET — Test Brier (reference only; AUC drives selection)")
print("=" * 70 + "\n")
pivot_brier = binary_df.pivot_table(
    index="feature_set", columns="split", values="test_brier"
)
pivot_brier["Mean"] = pivot_brier.mean(axis=1)
print(pivot_brier.round(4).to_string())

# ── Continuous R² ──
print("\n" + "=" * 70)
print("  CONTINUOUS TARGET — Test R² (percent space)")
print("=" * 70 + "\n")
pivot_r2 = cont_df.pivot_table(
    index="feature_set", columns="split", values="test_r2"
)
pivot_r2["Mean"] = pivot_r2.mean(axis=1)
print(pivot_r2.round(4).to_string())

# ── Continuous MSE ──
print("\n" + "=" * 70)
print("  CONTINUOUS TARGET — Test MSE (percent², comparable to the vol baseline)")
print("=" * 70 + "\n")
pivot_mse = cont_df.pivot_table(
    index="feature_set", columns="split", values="test_mse"
)
pivot_mse["Mean"] = pivot_mse.mean(axis=1)
print(pivot_mse.round(4).to_string())

# ── Derived AUC ──
print("\n" + "=" * 70)
print("  CONTINUOUS TARGET — Derived AUC")
print("  (rank-based: roc_auc_score(y_binary, -y_pred))")
print("  Directly comparable to binary AUC across all models.")
print("=" * 70 + "\n")
pivot_dauc = cont_df.pivot_table(
    index="feature_set", columns="split", values="test_derived_auc"
)
pivot_dauc["Mean"] = pivot_dauc.mean(axis=1)
print(pivot_dauc.round(4).to_string())

# ── Prediction std sanity check ──
# The target has a standard deviation of roughly 1.2 percentage points, so a
# model carrying any signal should produce a prediction std around 0.3–0.5.
# The threshold below is calibrated for that scale, not for a unit-variance
# z-scored target.
print("\n" + "=" * 70)
print("  CONTINUOUS SANITY: Prediction std (should be > 0.1 in percent space)")
print("=" * 70 + "\n")
for _, row in cont_df.iterrows():
    flag = "⚠ NEAR CONSTANT" if row.get("test_pred_std", 1) < 0.1 else "✓"
    print(f"  {row['feature_set']:<20} {row['split']:<10}  "
          f"pred_std={row.get('test_pred_std', 0):.4f}  {flag}")

# ── Overfitting check ──
print("\n" + "=" * 70)
print("  OVERFITTING CHECK — Train vs Test (binary)")
print("=" * 70 + "\n")
for _, row in binary_df.iterrows():
    gap  = row["train_auc"] - row["test_auc"]
    flag = " ⚠" if gap > 0.10 else ""
    print(f"  {row['feature_set']:<20} {row['split']:<10} "
          f"Train={row['train_auc']:.4f}  Test={row['test_auc']:.4f}  "
          f"Gap={gap:+.4f}{flag}")

print("\n" + "=" * 70)
print("  OVERFITTING CHECK — Train vs Test (continuous R²)")
print("=" * 70 + "\n")
for _, row in cont_df.iterrows():
    gap  = row["train_r2"] - row["test_r2"]
    flag = " ⚠" if gap > 0.15 else ""
    print(f"  {row['feature_set']:<20} {row['split']:<10} "
          f"Train={row['train_r2']:.4f}  Test={row['test_r2']:.4f}  "
          f"Gap={gap:+.4f}{flag}")

# ── Best hyperparameters ──
print("\n" + "=" * 70)
print("  BEST HYPERPARAMETERS")
print("=" * 70 + "\n")
for _, row in results_df.iterrows():
    print(f"  {row['feature_set']:<20} {row['split']:<10} "
          f"{row['target']:<12} {row['best_param']}")

# %% [markdown]
# ## Backtests
#
# `run_full_backtest` handles everything in one call per feature-set × split:
#   - Signal diagnostics (raw vs smoothed range, grid-edge warnings)
#   - Simple timing: threshold chosen on val (net-of-cost Sortino), test results
#   - Simple timing cost sensitivity (threshold re-chosen at each cost level)
#   - Asymmetric risk-scaled: params found on val, test results
#   - Risk-scaled cost sensitivity (params re-found at each cost level)
#   - Side-by-side comparison of both strategies vs buy-and-hold
#
# All backtests use 3 bps transaction costs and EMA span 5 by default,
# matching the global defaults set in evaluation.py.
#
# Binary signal    : y_prob (go to cash when prob is HIGH → go_cash_when="above")
# Continuous signal: y_pred (go to cash when predicted drawdown is LOW, i.e.
#                    more negative → go_cash_when="below")

# %% [markdown]
# ### Binary Signal Backtests
# %%
binary_backtest_results = {}
for feature_set in FEATURE_SETS:
    for split_name in SPLITS:
        loaded_val  = load_predictions(
            f"ridge_{feature_set}", split_name, "binary", "val",
            results_dir=RESULTS_DIR,
        )
        loaded_test = load_predictions(
            f"ridge_{feature_set}", split_name, "binary", "test",
            results_dir=RESULTS_DIR,
        )
        res = run_full_backtest(
            val_returns  = loaded_val["predictions"]["daily_return"].values,
            val_signal   = loaded_val["predictions"]["y_prob"].values,
            test_returns = loaded_test["predictions"]["daily_return"].values,
            test_signal  = loaded_test["predictions"]["y_prob"].values,
            go_cash_when = "above",
            model_name   = f"ridge_{feature_set}",
            split_name   = split_name,
        )
        binary_backtest_results[(feature_set, split_name)] = res

# %% [markdown]
# ### Continuous Signal Backtests
# %%
cont_backtest_results = {}
for feature_set in FEATURE_SETS:
    for split_name in SPLITS:
        loaded_val  = load_predictions(
            f"ridge_{feature_set}", split_name, "continuous", "val",
            results_dir=RESULTS_DIR,
        )
        loaded_test = load_predictions(
            f"ridge_{feature_set}", split_name, "continuous", "test",
            results_dir=RESULTS_DIR,
        )
        res = run_full_backtest(
            val_returns  = loaded_val["predictions"]["daily_return"].values,
            val_signal   = loaded_val["predictions"]["y_pred"].values,
            test_returns = loaded_test["predictions"]["daily_return"].values,
            test_signal  = loaded_test["predictions"]["y_pred"].values,
            go_cash_when = "below",
            model_name   = f"ridge_{feature_set}",
            split_name   = split_name,
        )
        cont_backtest_results[(feature_set, split_name)] = res

# %% [markdown]
# ### Cross-Split Backtest Summary
#
# Pulls the headline numbers out of the stored results dicts and assembles
# a compact comparison table — useful for the dissertation results section.
# %%
print("\n" + "=" * 70)
print("  BINARY SIGNAL — Headline Backtest Summary (3 bps, EMA 5)")
print("=" * 70)
print(f"\n  {'Feature set':<20} {'Split':<10} {'Strat':<22} "
      f"{'Sharpe':>8} {'Sortino':>8} {'AnnRet':>8} {'MaxDD':>8} {'AvgExp':>7}")
print("  " + "-" * 92)
for feature_set in FEATURE_SETS:
    for split_name in SPLITS:
        res = binary_backtest_results[(feature_set, split_name)]
        for label, d, exp in [
            ("Simple timing",   res["simple"],      res["simple"]["avg_exposure"]),
            ("Risk-scaled",     res["risk_scaled"], res["risk_scaled"]["avg_exposure"]),
            ("Buy & Hold",      res["buy_hold"],    1.0),
        ]:
            srt = f"{d['sortino']:>8.2f}" if np.isfinite(d["sortino"]) else "     nan"
            print(f"  {feature_set:<20} {split_name:<10} {label:<22} "
                  f"{d['sharpe']:>8.2f} {srt} "
                  f"{d['annual_return']:>8.1%} {d['max_drawdown']:>8.1%} "
                  f"{exp:>6.1%}")
    print()

print("\n" + "=" * 70)
print("  CONTINUOUS SIGNAL — Headline Backtest Summary (3 bps, EMA 5)")
print("=" * 70)
print(f"\n  {'Feature set':<20} {'Split':<10} {'Strat':<22} "
      f"{'Sharpe':>8} {'Sortino':>8} {'AnnRet':>8} {'MaxDD':>8} {'AvgExp':>7}")
print("  " + "-" * 92)
for feature_set in FEATURE_SETS:
    for split_name in SPLITS:
        res = cont_backtest_results[(feature_set, split_name)]
        for label, d, exp in [
            ("Simple timing",   res["simple"],      res["simple"]["avg_exposure"]),
            ("Risk-scaled",     res["risk_scaled"], res["risk_scaled"]["avg_exposure"]),
            ("Buy & Hold",      res["buy_hold"],    1.0),
        ]:
            srt = f"{d['sortino']:>8.2f}" if np.isfinite(d["sortino"]) else "     nan"
            print(f"  {feature_set:<20} {split_name:<10} {label:<22} "
                  f"{d['sharpe']:>8.2f} {srt} "
                  f"{d['annual_return']:>8.1%} {d['max_drawdown']:>8.1%} "
                  f"{exp:>6.1%}")
    print()

# %% [markdown]
# ## File Inventory
# %%
print("\n" + "=" * 70)
print("  SAVED FILES")
print("=" * 70 + "\n")

pred_dir    = RESULTS_DIR / "predictions"
metrics_dir = RESULTS_DIR / "metrics"

if pred_dir.exists():
    pred_files = sorted(pred_dir.glob("ridge_*.parquet"))
    print(f"  Predictions: {len(pred_files)} files")
    for f in pred_files[:6]:
        print(f"    {f.name}")
    if len(pred_files) > 6:
        print(f"    ... and {len(pred_files) - 6} more")

if metrics_dir.exists():
    metric_files = sorted(metrics_dir.glob("ridge_*.json"))
    print(f"\n  Metrics: {len(metric_files)} files")

# 4 splits × 2 feature sets × 2 targets × 3 parts = 48 prediction files
expected = len(SPLITS) * len(FEATURE_SETS) * len(TARGET_TYPES) * 3
print(f"\n  Expected: {expected} prediction files + {expected} metric files")
if pred_dir.exists():
    actual = len(sorted(pred_dir.glob("ridge_*.parquet")))
    status = "✓" if actual == expected else f"⚠ got {actual}"
    print(f"  Actual:   {actual}  {status}")
# %%

  RIDGE BASELINE: 4 Splits × 2 Feature Sets × 2 Targets
  Continuous target: minret_5d_pct (percentage points, NOT z-scored)
  Continuous selection: maximise val R²
  Binary selection: maximise val AUC (uniform selection metric across
  every model in this project — Ridge, Dense/Sparse MLP, Dense/Sparse KAN)
  Derived AUC: roc_auc_score(y_binary, -y_pred) — rank-based, no threshold

  agg_full_moments / Split_A / binary
    Best C=0.00050  |  Val AUC=0.6852  |  Val Brier=0.0640  |  Test AUC=0.6828  |  Gap=+0.2351
  agg_full_moments / Split_A / continuous
    Best α=30000  |  Val R²=-0.2478  |  Test R²=0.1643  |  Test MSE=0.7932  |  Derived AUC=0.8421  |  Pred std=0.2352

  agg_full_moments / Split_B / binary
    Best C=0.00001  |  Val AUC=0.7808  |  Val Brier=0.1550  |  Test AUC=0.7275  |  Gap=+0.1112
  agg_full_moments / Split_B / continuous
    Best α=100000  |  Val R²=0.0965  |  Test R²=0.1482  |  Test MSE=2.3869  |  Derived AUC=0.7311  |  Pred std=0.3643

  agg_full_moments / Split